# Arsitrad — Gemma 4 Inference Demo
## End-to-End: RAG + Fine-tuned Gemma 4 2B → Gradio UI

**Runtime:** Colab with **2x T4 GPU** (High-RAM recommended)

---


## 1. Setup & Installation


In [ ]:
#@title 1.2 — Install dependencies
!pip install -q --upgrade pip
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes \
    accelerate \
    scipy \
    chromadb>=0.4.0 \
    sentence-transformers>=2.5.0 \
    gradio>=4.0.0 \
    huggingface_hub \
    tqdm
!pip install -q unsloth
print('All dependencies installed!')


In [ ]:
#@title 1.3 — Imports & Config
import os
import torch
from pathlib import Path

BASE_DIR   = Path('/content/arsitrad')
LORA_DIR   = BASE_DIR / 'lora_adapters'
CHROMA_DIR = BASE_DIR / 'chroma_db'

RAG_CONFIG = {
    'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'collection_name': 'arsitrad_national_regulations',
    'top_k': 5,
    'min_similarity': 0.3,
}

MODEL_CONFIG = {
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'device_map': 'auto',                       # shards base model across all GPUs
    'base_model': 'google/gemma-4-e2b-it',      # full base model for device_map="auto"
}
print('Config ready.')


## 2. Download LoRA Adapters + ChromaDB


In [ ]:
#@title 2.1 — Create directories
!mkdir -p /content/arsitrad/lora_adapters
!mkdir -p /content/arsitrad/chroma_db
print('Directories created.')


In [ ]:
#@title 2.2 — Download LoRA adapters from GitHub Release
import os, subprocess

LORA_URL = 'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/arsitrad_finetuned_adapters.zip'
out = '/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters.zip'

if os.path.exists('/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters'):
    print('LoRA adapters already exist, skipping download.')
else:
    print('Downloading LoRA adapters (~747 MB)...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', out, LORA_URL], check=True)
    print('Unzipping...')
    subprocess.run(['unzip', '-q', '-o', out, '-d', '/content/arsitrad/lora_adapters/'], check=True)
    print('Done!')


In [ ]:
#@title 2.3 — Download and Fix ChromaDB Extraction
import os, subprocess, shutil

CHROMA_ZIP = '/content/arsitrad/chroma_db_release.zip'
TARGET_DIR = '/content/arsitrad/chroma_db'
TEMP_EXTRACT = '/content/arsitrad/temp_chroma'

# Clean up previous attempts
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
if os.path.exists(TEMP_EXTRACT): shutil.rmtree(TEMP_EXTRACT)
os.makedirs(TARGET_DIR, exist_ok=True)

print('Downloading ChromaDB (~262 MB)...')
subprocess.run(['wget', '-q', '--show-progress', '-O', CHROMA_ZIP, 'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/chroma_db_release.zip'], check=True)

print('Extracting to temporary folder...')
subprocess.run(['unzip', '-q', '-o', CHROMA_ZIP, '-d', TEMP_EXTRACT], check=True)

# Find where the actual sqlite file is (handling nested folders)
sqlite_source_dir = None
for root, dirs, files in os.walk(TEMP_EXTRACT):
    if 'chroma.sqlite3' in files:
        sqlite_source_dir = root
        break

if sqlite_source_dir:
    print(f'Found database in {sqlite_source_dir}. Moving to {TARGET_DIR}...')
    for item in os.listdir(sqlite_source_dir):
        shutil.move(os.path.join(sqlite_source_dir, item), os.path.join(TARGET_DIR, item))
    print('Done!')
else:
    print('Error: chroma.sqlite3 not found in the downloaded ZIP.')

# Cleanup
shutil.rmtree(TEMP_EXTRACT)
if os.path.exists(CHROMA_ZIP): os.remove(CHROMA_ZIP)
print('Final Contents of', TARGET_DIR, ':', os.listdir(TARGET_DIR))

## 3. RAG Retrieval Pipeline


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
import importlib

# Force a clean state at the library level
importlib.reload(chromadb)
from chromadb.api.shared_system_client import SharedSystemClient
SharedSystemClient.clear_system_cache()

class RegulationRetriever:
    def __init__(self, persist_dir, collection_name):
        self.emb_model = SentenceTransformer(RAG_CONFIG['embedding_model'])
        # Using standard settings to avoid conflict with existing cached systems
        self.client = chromadb.PersistentClient(path=persist_dir)

        try:
            # Try to get existing collection
            self.collection = self.client.get_collection(name=collection_name)
            print(f'✅ Success! Collection "{collection_name}" loaded with {self.collection.count()} chunks.')
        except Exception as e:
            print(f'❌ Failed to get collection: {e}')
            cols = self.client.list_collections()
            print('Available collections:', [c.name for c in cols] if cols else 'None')
            raise e

    def retrieve(self, query, top_k=5, min_similarity=0.3):
        raw_emb = self.emb_model.encode([query])[0]
        q_norm = raw_emb / (np.linalg.norm(raw_emb) + 1e-8)
        results = self.collection.query(
            query_embeddings=[q_norm.tolist()],
            n_results=top_k,
            include=['documents','metadatas','embeddings'],
        )
        outputs = []
        for doc, metadata, emb in zip(results['documents'][0],
                                      results['metadatas'][0],
                                      results['embeddings'][0]):
            if emb is None: continue
            cos_sim = float(np.dot(q_norm, np.array(emb)) / (np.linalg.norm(np.array(emb)) + 1e-8))
            if cos_sim >= min_similarity:
                outputs.append({
                    'text': doc,
                    'source': metadata.get('source','unknown'),
                    'chunk_id': metadata.get('chunk_id',0),
                    'similarity': round(cos_sim,4)
                })
        outputs.sort(key=lambda x: x['similarity'], reverse=True)
        return outputs

    def format_context(self, retrieved):
        if not retrieved:
            return 'No relevant regulations found.'
        parts = []
        for i, item in enumerate(retrieved, 1):
            parts.append(f"[{i}] {item['text']}\n(Source: {item['source']}, chunk {item['chunk_id']})")
        return '\n\n'.join(parts)

# Re-initialize after clearing cache
retriever = RegulationRetriever(str(CHROMA_DIR), RAG_CONFIG['collection_name'])

In [ ]:
import sqlite3

db_path = '/content/arsitrad/chroma_db/chroma.sqlite3'
conn = sqlite3.connect(db_path)
cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables:", [r[0] for r in cursor.fetchall()])

try:
    cursor2 = conn.execute("SELECT id, name FROM collections;")
    print("Collections:", cursor2.fetchall())
except Exception as e:
    print(f"Error: {e}")
conn.close()

In [ ]:
!pip install -q --upgrade chromadb
import chromadb
print(f"Updated ChromaDB to version: {chromadb.__version__}")

In [ ]:
import chromadb
import importlib
importlib.reload(chromadb)

# Initialize client with the absolute path
client = chromadb.PersistentClient(path="/content/arsitrad/chroma_db")

try:
    # Attempt to fetch the specific collection
    coll = client.get_collection("arsitrad_national_regulations")
    print(f"✅ Success! Collection 'arsitrad_national_regulations' found.")
    print(f"📊 Total chunks loaded: {coll.count()}")
except Exception as e:
    print(f"❌ Error retrieving collection: {e}")
    print("\nAvailable collections in this database:")
    collections = client.list_collections()
    if not collections:
        print("  (No collections found)")
    for c in collections:
        print(f"  - {c.name}")

In [ ]:
import os

# Check what's actually in the chroma_db directory
if os.path.exists('/content/arsitrad/chroma_db/'):
    print("Contents of /content/arsitrad/chroma_db/:")
    for f in os.listdir('/content/arsitrad/chroma_db/'):
        file_path = f'/content/arsitrad/chroma_db/{f}'
        size = os.path.getsize(file_path)
        print(f"  {f} — {size/1e6:.1f} MB")
else:
    print("Directory /content/arsitrad/chroma_db/ does not exist.")

In [ ]:
import sqlite3
import os

db_path = '/content/arsitrad/chroma_db/chroma.sqlite3'
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [r[0] for r in cursor.fetchall()]
    print("Tables in ChromaDB:", tables)

    if 'collections' in tables:
        cursor2 = conn.execute("SELECT id, name FROM collections;")
        rows = cursor2.fetchall()
        print(f"Collections found in SQLite: {rows}")
    else:
        print("CRITICAL: 'collections' table missing from SQLite file.")
    conn.close()
else:
    print(f"SQLite file not found at {db_path}")

In [ ]:
#@title 3.2 — Test RAG retrieval
test_query = 'syarat teknis ventilasi alami pada rumah tinggal'
results   = retriever.retrieve(test_query, top_k=3)

print(f'Query: {test_query}\n')
for r in results:
    print(f"  [{r['similarity']:.3f}] {r['source']} — {r['text'][:120]}...")
print(f'\nFormatted context:\n{retriever.format_context(results)[:400]}')


In [ ]:
#@title 3.3 — Fix missing adapter_config.json in LoRA folder
import json, os

adapter_config = {
    "base_model_name_or_path": "google/gemma-4-e2b-it",
    "bias": "none",
    "inference_mode": True,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": 64,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    "task_type": "CAUSAL_LM",
}

LORA_PATH = "/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters"
os.makedirs(LORA_PATH, exist_ok=True)

with open(os.path.join(LORA_PATH, "adapter_config.json"), "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"adapter_config.json written — {LORA_PATH}")


## 4. Load Fine-tuned Gemma 4 + Inference


In [ ]:
# @title 4.1 — Load Gemma 4 2B + LoRA (Unsloth, Kaggle 2x T4)
# Restart your kernel before running to clear previous imports!

# 1. IMPORT UNSLOTH FIRST
import unsloth
import json, os
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

BASE_MODEL = "google/gemma-4-e2b-it"
lora_path = str(LORA_DIR)

# Step 1: Write adapter_config.json if missing
adapter_config = {
    "base_model_name_or_path": BASE_MODEL,
    "bias": "none",
    "inference_mode": True,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": 64,
    "target_modules": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    "task_type": "CAUSAL_LM",
}

os.makedirs(LORA_PATH, exist_ok=True)
if not os.path.exists(os.path.join(LORA_PATH, "adapter_config.json")):
    with open(os.path.join(LORA_PATH, "adapter_config.json"), "w") as f:
        json.dump(adapter_config, f, indent=2)
    print("Created adapter_config.json")
else:
    print("adapter_config.json already exists")

total_mem = {i: int(torch.cuda.get_device_properties(i).total_memory) for i in range(torch.cuda.device_count())}
print(f"GPUs detected: {total_mem}")

# Step 2 & 3: Load Base Model AND LoRA Adapter simultaneously via Unsloth
# By passing LORA_PATH, Unsloth reads the config, downloads the base model, and attaches the adapters natively.
print(f"Loading model and LoRA from {LORA_PATH} with Unsloth (4-bit)...")

model, _ = FastLanguageModel.from_pretrained(
    model_name             = lora_path, # <--- CHANGED: Pass LORA_PATH directly here
    max_seq_length         = 2048,
    load_in_4bit           = True,
    dtype                  = None,
    device_map             = "auto",
    max_memory             = total_mem,
    gpu_memory_utilization = 0.7,   
)
print("Base model and LoRA adapters loaded.")

# Step 4: Enable 2x faster inference
FastLanguageModel.for_inference(model)

# Verify GPUs
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")

print(f"\n✅ Model ready! LoRA: {LORA_PATH}")

In [ ]:
#@title 4.2 — Build prompt with RAG context (with optional reasoning mode)
SYSTEM_PROMPT = (
    'Kamu adalah Arsitrad, asisten AI berlisensi untuk regulasi dan arsitektur Indonesia. '
    'Jawab secara komprehensif — gunakan konteks peraturan yang diberikan untuk memberikan '
    'jawaban yang detail, terstruktur, dan akurat. '
    'Sertakan rujukan spesifik (nama peraturan dan pasalnya) bila memungkinkan. '
    'Jika konteks tidak cukup, jawablah dengan pengetahuan Anda secara jujur dan jelaskan asumsinya.'
)

def build_prompt(user_query, context, use_reasoning=False):
    if context and context != 'No relevant regulations found.':
        full_query = (
            'Konteks peraturan (jawablah Merujuk ke konteks ini):\n'
            + context + '\n\n'
            'Pertanyaan pengguna:\n' + user_query
        )
    else:
        full_query = user_query

    prompt = (
        '<start_of_turn>system\n' + SYSTEM_PROMPT + '<end_of_turn>\n'
        '<start_of_turn>user\n' + full_query + '<end_of_turn>\n'
    )
    if use_reasoning:
        prompt += '<start_of_turn>thought\n'
    else:
        prompt += '<start_of_turn>model\n'
    return prompt

print("✅ Cell 4.2 loaded successfully!")

In [ ]:
#@title 4.3 — Streaming generator (Gemma 4 reasoning mode support)
from transformers import TextIteratorStreamer
from threading import Thread

def generate_response_stream(user_query,
                              max_new_tokens=512,
                              temperature=0.2,
                              top_p=0.95,
                              top_k=64,
                              repetition_penalty=1.05,
                              use_reasoning=False):
    retrieved  = retriever.retrieve(user_query, top_k=RAG_CONFIG['top_k'])
    context    = retriever.format_context(retrieved)
    prompt     = build_prompt(user_query, context, use_reasoning=use_reasoning)
    inputs     = tokenizer(prompt, return_tensors='pt', truncation=True)
    inputs     = {k: v.to(model.device) for k, v in inputs.items()}

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        streamer=streamer,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    accumulated  = ''
    thought_block = ''

    for new_text in streamer:
        accumulated += new_text

        if use_reasoning:
            t_start_tag = '<start_of_turn>thought\n'
            if t_start_tag in accumulated:
                t_start = accumulated.find(t_start_tag) + len(t_start_tag)
                t_end = accumulated.find('<end_of_turn>', t_start)
                if t_end != -1:
                    thought_block = accumulated[t_start:t_end]
                    after_thought = accumulated[t_end + len('<end_of_turn>'):]
                    final_answer = after_thought.replace('<start_of_turn>model\n', '', 1)
                    yield final_answer, context, retrieved, thought_block
                else:
                    yield '', context, retrieved, ''
            else:
                yield '', context, retrieved, ''
        else:
            yield accumulated, context, retrieved, ''

    thread.join()

print("✅ Cell 4.3 loaded successfully!")

In [ ]:
#@title 4.4 — Test inference with streaming + reasoning mode demo
import time

test_queries = [
    'Apa saja syarat IMB untuk rumah tinggal 2 lantai di Semarang?',
    'Bagaimana klasifikasi kerusakan bangunan akibat gempa bumi?',
    'Strategi passive cooling untuk rumah di daerah tropis basah?',
]

# Normal generation (reasoning=off)
for q in test_queries:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'{"-"*60}')
    print('R: ', end='', flush=True)

    accumulated = ''
    start = time.time()
    for partial_response, _, retrieved, _ in generate_response_stream(q, max_new_tokens=512, use_reasoning=False):
        new_tokens = partial_response[len(accumulated):]
        print(new_tokens, end='', flush=True)
        accumulated = partial_response
    elapsed = time.time() - start
    print(f'\n\n[time {elapsed:.1f}s | {len(accumulated)} chars | {len(retrieved)} chunks cited | reasoning=off]')

# Reasoning mode demo — first query only
print(f'\n\n{"="*60}')
print('REASONING MODE DEMO — first query')
print(f'{"="*60}')
q = test_queries[0]
print(f'Q: {q}')
print('R: ', end='', flush=True)

thought_acc = ''
accumulated = ''
start = time.time()
for partial_response, _, retrieved, thought in generate_response_stream(q, max_new_tokens=512, use_reasoning=True):
    new_tokens = partial_response[len(accumulated):]
    print(new_tokens, end='', flush=True)
    accumulated = partial_response
    if thought:
        thought_acc = thought
elapsed = time.time() - start

print(f'\n\n[time {elapsed:.1f}s | {len(accumulated)} chars]')
if thought_acc:
    print(f'[thought preview: {thought_acc[:300]}...]')

print('\nAll test queries passed!')


In [ ]:
# Force fastest inference mode
import torch

# Disable gradient tracking globally – not needed for inference
torch.set_grad_enabled(False)

# Verify model is in eval/ inference mode
model.eval()
print(f"Model training mode: {model.training}")
print(f"Inference optimizations applied. Please re-run the following cells.")

## 5. Gradio UI — Live Demo


In [ ]:
#@title 5.1 — Gradio UI: Cream/Charcoal Theme (Bulletproof Prose Fix)
import gradio as gr
import time

# --- 1. THEME DEFINITION (Light Theme) ---
theme = gr.themes.Base(
    font=[gr.themes.GoogleFont("Manrope"), "sans-serif"],
    font_mono=[gr.themes.GoogleFont("Inter"), "monospace"],
).set(
    body_background_fill="#faf9f5", 
    body_text_color="#141413",      
    block_background_fill="transparent",
    block_border_color="transparent",
    panel_background_fill="transparent",
    input_background_fill="#ffffff",
    input_border_color="#1f1e1d",   
    button_primary_background_fill="#141413", 
    button_primary_background_fill_hover="#333333",
    button_primary_text_color="#ffffff",
    slider_color="#2c84db",         
)

# --- 2. CSS OVERRIDES ---
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;600;700&family=Manrope:wght@300;400;600;700&family=Inter:wght@400;500;600;700&display=swap');

/* Global Reset */
footer { display: none !important; }
.gradio-container { background-color: #faf9f5 !important; max-width: 100% !important; padding: 0 !important; margin: 0 !important; }

/* Fix Backgrounds on Forms/Sliders */
.form, .wrap, .box, .container { background-color: transparent !important; border: none !important; box-shadow: none !important; }

/* Sidebar */
#sidebar { background-color: #faf9f5; border-right: 1px solid #1f1e1d; height: 100vh; padding: 32px 24px; position: sticky; top: 0; }

/* Main Canvas Layout */
#main-canvas { padding: 0 5% !important; }

/* Typography */
.font-outfit { font-family: 'Outfit', sans-serif !important; }
.font-inter { font-family: 'Inter', sans-serif !important; }
.tracking-widest { letter-spacing: 0.1em; text-transform: uppercase; }

/* Clean Chatbox */
#chat-window { height: calc(100vh - 250px) !important; overflow-y: auto; }
.message-wrap { gap: 16px !important; padding: 10px 0 !important; }
.message.user { background: #ffffff !important; border: 1px solid #1f1e1d !important; border-radius: 16px !important; color: #141413 !important; padding: 14px 18px !important; box-shadow: 2px 2px 0px rgba(0,0,0,0.05) !important; }
.message.bot { background: transparent !important; border: none !important; color: #141413 !important; padding: 12px 4px !important; font-size: 15.5px !important; line-height: 1.6 !important; }

/* 🚨 BULLETPROOF PROSE FIX 🚨 */
/* 1. Override Tailwind CSS Variables */
.prose, .chatbot.prose {
    --tw-prose-body: #141413 !important;
    --tw-prose-headings: #141413 !important;
    --tw-prose-lead: #141413 !important;
    --tw-prose-links: #2c84db !important;
    --tw-prose-bold: #000000 !important;
    --tw-prose-counters: #141413 !important;
    --tw-prose-bullets: #141413 !important;
    --tw-prose-quotes: #141413 !important;
    --tw-prose-code: #1f1e1d !important;
    color: #141413 !important;
}

/* 2. Brute Force HTML Tags */
.message.bot p, .message.bot li, .message.bot h1, .message.bot h2, .message.bot h3, .message.bot span, div[data-testid="bot"] .prose * {
    color: #141413 !important;
}
.message.bot strong { color: #000000 !important; font-weight: 700 !important; }

/* Custom Input Container */
#input-container { background-color: #ffffff !important; border: 1px solid #1f1e1d !important; border-radius: 16px !important; padding: 16px !important; margin-top: 10px; box-shadow: 2px 2px 0px rgba(0,0,0,0.05) !important; }
#input-box textarea { color: #141413 !important; }
#action-row { align-items: center !important; margin-top: 12px !important; padding-top: 12px !important; border-top: 1px solid #e0e0e0 !important; justify-content: space-between !important; }

/* Buttons & Checkboxes */
#save-btn { font-family: 'Outfit', sans-serif !important; font-weight: 700 !important; letter-spacing: 0.1em !important; text-transform: uppercase !important; border-radius: 8px !important; }
#clear-btn { background: transparent !important; color: #666666 !important; border: none !important; box-shadow: none !important; font-family: 'Outfit', sans-serif !important; font-weight: 700 !important; letter-spacing: 0.1em !important; text-transform: uppercase !important; }
#clear-btn:hover { color: #141413 !important; }

/* Focus States (Vibrant Blue Accent) */
textarea:focus, input:focus { border-color: #2c84db !important; outline: none !important; }

/* Label Colors */
span.svelte-1gfkn6j { color: #141413 !important; font-weight: 600 !important; }

/* Custom Scrollbar */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: #d1d0cc; border-radius: 5px; }
::-webkit-scrollbar-thumb:hover { background: #b0afab; }
"""

# --- 3. MULTIMODAL STREAMING LOGIC ---
def add_user_message(user_input, history):
    history = history or []
    text = user_input["text"]
    files = user_input["files"]
    
    # Render files in the chat as tuples
    for f in files:
        history.append([(f,), None])
        
    # Render text in the chat
    if text:
        history.append([text, None])
    elif not text and files:
        history.append(["[File Attached]", None])
        
    # Clear the input box
    return gr.MultimodalTextbox(value={"text": "", "files": []}), history

def arsitrad_stream(history, max_tokens, temperature, top_p, top_k, use_reasoning):
    user_message = history[-1][0]
    
    # Fallback if user only sends a file with no text
    if isinstance(user_message, tuple) or user_message == "[File Attached]":
        user_message = "Tolong analisis file yang saya lampirkan."
    
    for partial_response, _, retrieved, thought in generate_response_stream(
        user_message,
        max_new_tokens=int(max_tokens),
        temperature=float(temperature),
        top_p=float(top_p),
        top_k=int(top_k),
        use_reasoning=bool(use_reasoning),
    ):
        # Format explicitly for light theme
        sources_html = ""
        if retrieved:
            sources_html = '<div style="margin-top:20px; padding:16px; background:#ffffff; border:1px solid #1f1e1d; border-radius:12px; font-family:\'Inter\', monospace; font-size:12px; color:#141413;">'
            sources_html += '<div style="margin-bottom:8px; color:#666666; font-weight:600;">// Retrieved Context</div>'
            for r in retrieved:
                 sources_html += f'<div style="margin-bottom:4px; border-bottom:1px solid #f0f0f0; padding-bottom:4px;">[<span style="color:#2c84db; font-weight:bold;">{r["similarity"]:.2f}</span>] {r["source"]}</div>'
            sources_html += '</div>'
            
        thought_html = ""
        if thought:
            thought_html = '<div style="margin-bottom:20px; padding:16px; background:#ffffff; border:1px solid #1f1e1d; border-left:4px solid #2c84db; border-radius:8px; font-family:\'Manrope\'; font-size:14px; color:#333333;">'
            thought_html += '<div class="font-inter tracking-widest" style="font-size:10px; margin-bottom:8px; color:#2c84db; font-weight:bold;">Reasoning Engine</div>'
            thought_html += thought.strip() + '</div>'
            
        full_bot_msg = (partial_response if partial_response else '') + thought_html + sources_html
        history[-1][1] = full_bot_msg
        yield history

# --- 4. LAYOUT ---
with gr.Blocks(theme=theme, css=custom_css, title="Arsitrad Slate") as demo:
    
    with gr.Row(equal_height=True):
        
        # === LEFT SIDEBAR ===
        with gr.Column(scale=1, min_width=280, elem_id="sidebar"):
            gr.HTML("""
            <div class="font-outfit" style="font-size:2rem; font-weight:700; color:#141413; margin-bottom:40px; margin-top:10px;">
                Arsitrad
            </div>
            """)
            
            temp_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.7, step=0.05, label="TEMPERATURE")
            top_p_slider = gr.Slider(minimum=0.5, maximum=1.0, value=0.9, step=0.01, label="TOP-P")
            top_k_slider = gr.Slider(minimum=0, maximum=256, value=64, step=8, label="TOP-K") 
            max_tok_slider = gr.Slider(minimum=128, maximum=1024, value=512, step=64, label="MAX TOKENS")
            
            gr.HTML("<div style='margin-bottom: 20px;'></div>")
            gr.Button("SAVE PRESET", variant="primary", elem_id="save-btn", size="sm")

        # === MAIN CANVAS ===
        with gr.Column(scale=4, elem_id="main-canvas"):
            
            gr.HTML("""
            <div style="display:flex; justify-content:flex-end; align-items:center; padding:24px 0; margin-bottom:10px;">
                <div class="font-inter tracking-widest" style="color:#666666; font-size:11px; font-weight:600;">STATUS: GPU ACTIVE</div>
            </div>
            """)
            
            chatbot = gr.Chatbot(
                show_label=False,
                elem_id="chat-window",
                type="tuples",
            )
            
            with gr.Group(elem_id="input-container"):
                msg_input = gr.MultimodalTextbox(
                    file_types=["image", ".pdf", ".txt"],
                    placeholder="Submit your prompt here...",
                    show_label=False,
                    lines=2,
                )
                
                with gr.Row(elem_id="action-row"):
                    reasoning_toggle = gr.Checkbox(label="REASONING MODE", value=False)
                    clear_btn = gr.Button("CLEAR", elem_id="clear-btn", scale=0) 
            
            gr.HTML("""
            <div style="text-align:center; margin-top:20px; margin-bottom:40px;">
                <span class="font-inter tracking-widest" style="color:#888888; font-size:10px;">POWERED BY GEMMA 4 E2B-IT • HIGH FIDELITY MODE</span>
            </div>
            """)
            
    # --- EVENT ROUTING ---
    msg_input.submit(
        add_user_message, [msg_input, chatbot], [msg_input, chatbot], queue=False
    ).then(
        arsitrad_stream, [chatbot, max_tok_slider, temp_slider, top_p_slider, top_k_slider, reasoning_toggle], chatbot
    )
    
    clear_btn.click(lambda: None, None, chatbot, queue=False)

print("Ready — Cream Theme deployed with Bulletproof Prose text fix.")
demo.launch(server_name="0.0.0.0", server_port=7854, share=True)

---
## Troubleshooting

| Problem | Fix |
|---|---|
| OOM on T4 | Runtime → Restart runtime before inference (clears GPU memory from downloads) |
| Model not loading | Use LoRA path directly, or `google/gemma-4-E2B-it` for base model |
| ChromaDB empty | Re-run Section 2.3 — wget may timeout |
| Slow retrieval | Reduce `top_k` in RAG_CONFIG (e.g. 3 instead of 5) |
| Gradio link not working | Wait 30s after launch for Colab to provision the URL |
